# Notebook 1: Basic Spatial Transcriptomics Analysis with Scanpy

**Dataset:** 10x Genomics Visium - Human Lymph Node (V1_Human_Lymph_Node)

**Reference:** [Scanpy Spatial Tutorial](https://scanpy-tutorials.readthedocs.io/en/latest/spatial/basic-analysis.html)

This notebook demonstrates how to work with spatial transcriptomics data (Visium) within Scanpy, including QC, preprocessing, clustering, spatial visualization, and marker gene analysis. A MERFISH example is also included.

## 1. Setup & Imports

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns

sc.logging.print_versions()
sc.set_figure_params(facecolor="white", figsize=(8, 8))
sc.settings.verbosity = 3

## 2. Reading the Data

We load the Visium spatial transcriptomics dataset of the human lymph node from 10x Genomics.

In [ ]:
adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
adata.var_names_make_unique()
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

In [ ]:
# Inspect the AnnData structure
adata

## 3. QC and Preprocessing

We visualize QC metrics and filter spots based on total counts, expressed genes, and mitochondrial content.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))
sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0])
sns.histplot(
    adata.obs["total_counts"][adata.obs["total_counts"] < 10000],
    kde=False, bins=40, ax=axs[1],
)
sns.histplot(adata.obs["n_genes_by_counts"], kde=False, bins=60, ax=axs[2])
sns.histplot(
    adata.obs["n_genes_by_counts"][adata.obs["n_genes_by_counts"] < 4000],
    kde=False, bins=60, ax=axs[3],
)
plt.tight_layout()
plt.savefig("../results/01_basic_scanpy/01_qc_histograms.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sc.pp.filter_cells(adata, min_counts=5000)
sc.pp.filter_cells(adata, max_counts=35000)
adata = adata[adata.obs["pct_counts_mt"] < 20].copy()
print(f"#cells after MT filter: {adata.n_obs}")
sc.pp.filter_genes(adata, min_cells=10)

### Normalization and Highly Variable Gene Detection

In [ ]:
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=2000)

## 4. Manifold Embedding and Clustering

Standard PCA → neighbors → UMAP → Leiden clustering pipeline.

In [ ]:
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(
    adata, key_added="clusters", flavor="igraph",
    directed=False, n_iterations=2
)

In [ ]:
plt.rcParams["figure.figsize"] = (4, 4)
sc.pl.umap(adata, color=["total_counts", "n_genes_by_counts", "clusters"], wspace=0.4)
plt.savefig("../results/01_basic_scanpy/02_umap_covariates.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Visualization in Spatial Coordinates

Overlay total counts and gene counts on the H&E tissue image.

In [ ]:
plt.rcParams["figure.figsize"] = (8, 8)
sc.pl.spatial(adata, img_key="hires", color=["total_counts", "n_genes_by_counts"])
plt.savefig("../results/01_basic_scanpy/03_spatial_counts.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sc.pl.spatial(adata, img_key="hires", color="clusters", size=1.5)
plt.savefig("../results/01_basic_scanpy/04_spatial_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

### Zoomed View of Specific Clusters

In [ ]:
sc.pl.spatial(
    adata,
    img_key="hires",
    color="clusters",
    groups=["5", "9"],
    crop_coord=[7000, 10000, 0, 6000],
    alpha=0.5,
    size=1.3,
)
plt.savefig("../results/01_basic_scanpy/05_spatial_zoomed.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Cluster Marker Genes

Compute marker genes for cluster 9 and visualize with a heatmap.

In [ ]:
sc.tl.rank_genes_groups(adata, "clusters", method="t-test")
sc.pl.rank_genes_groups_heatmap(adata, groups="9", n_genes=10, groupby="clusters")
plt.savefig("../results/01_basic_scanpy/06_marker_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sc.pl.spatial(adata, img_key="hires", color=["clusters", "CR2"])
plt.savefig("../results/01_basic_scanpy/07_spatial_cr2.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sc.pl.spatial(adata, img_key="hires", color=["COL1A2", "SYPL1"], alpha=0.7)
plt.savefig("../results/01_basic_scanpy/08_spatial_genes.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. MERFISH Example

Demonstrating spatial analysis with FISH-based data (Xia et al. 2019).

In [ ]:
# Note: Download data files first
# coordinates: pnas.1912459116.sd15.xlsx
# counts: pnas.1912459116.sd12.csv
#
# Uncomment below to run:
# coordinates = pd.read_excel("./data/pnas.1912459116.sd15.xlsx", index_col=0)
# counts = sc.read_csv("./data/pnas.1912459116.sd12.csv").transpose()
# adata_merfish = counts[coordinates.index, :].copy()
# adata_merfish.obsm["spatial"] = coordinates.to_numpy()

In [ ]:
# sc.pp.normalize_per_cell(adata_merfish, counts_per_cell_after=1e6)
# sc.pp.log1p(adata_merfish)
# sc.pp.pca(adata_merfish, n_comps=15)
# sc.pp.neighbors(adata_merfish)
# sc.tl.umap(adata_merfish)
# sc.tl.leiden(
#     adata_merfish,
#     key_added="clusters",
#     resolution=0.5,
#     n_iterations=2,
#     flavor="igraph",
#     directed=False,
# )

In [ ]:
# sc.pl.umap(adata_merfish, color="clusters")
# sc.pl.embedding(adata_merfish, basis="spatial", color="clusters")
# plt.savefig("../results/01_basic_scanpy/09_merfish.png", dpi=150, bbox_inches="tight")
# plt.show()

## Summary

- Loaded and preprocessed Visium human lymph node data
- Performed QC filtering (counts, genes, MT content)
- Normalized and detected highly variable genes
- Clustered cells using Leiden algorithm → 10 clusters
- Visualized clusters in both UMAP and spatial coordinates
- Identified marker genes (e.g., CR2 for cluster 9)
- Demonstrated MERFISH spatial analysis workflow